# RiskMatch V3 — Quantitative Research Lab

One notebook for the whole project. The 27 chapter numbers are navigation anchors. Execution follows the [roadmap](../docs/ROADMAP.md), [A–Z checklist](../CHECKLIST_A_Z.md) and [prerequisite registry](../docs/TASK_DEPENDENCIES.csv).

Start with chapters 01 and 12 (mandate and 11 objectives) and 02 (Discovery), alongside data work. Validate the minimal ETF backtest (BT00, chapter 23) before advanced models. Test Explore on baselines, then improve and validate all candidates before final matching.

## Table of Contents
- [01 — Client Mandate](#chapitre-01)
- [02 — Behavioral Discovery](#chapitre-02)
- [03 — US Universe](#chapitre-03)
- [04 — Sources and Licenses](#chapitre-04)
- [05 — Point-in-Time Ingestion](#chapitre-05)
- [06 — Returns and FX](#chapitre-06)
- [07 — Descriptive Statistics](#chapitre-07)
- [08 — Return Forecasting](#chapitre-08)
- [09 — Volatility and Covariance](#chapitre-09)
- [10 — Factors and Regressions](#chapitre-10)
- [11 — Tail Risk](#chapitre-11)
- [12 — Financial Objectives](#chapitre-12)
- [13 — Baselines](#chapitre-13)
- [14 — Mean-Variance](#chapitre-14)
- [15 — Black-Litterman](#chapitre-15)
- [16 — Risk Parity and Hierarchical Methods](#chapitre-16)
- [17 — Downside Optimization](#chapitre-17)
- [18 — Advanced Optimization](#chapitre-18)
- [19 — Passive and Active Strategies](#chapitre-19)
- [20 — Alternative Strategies](#chapitre-20)
- [21 — US Treasuries](#chapitre-21)
- [22 — Futures](#chapitre-22)
- [23 — Walk-Forward Backtesting](#chapitre-23)
- [24 — Stress and Monte Carlo](#chapitre-24)
- [25 — Improvement, Matching and Comparison](#chapitre-25)
- [26 — Dynamic Management](#chapitre-26)
- [27 — Integration and Validation](#chapitre-27)

## Shared Setup
Run this cell before starting.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'riskmatch').exists())
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))

### INFRA.01 — Environment validation (2026-11-24)

**Observed baseline:** macOS arm64, Python 3.12.6, pip 24.2, Node 23.11.1. The active virtual environment inherits system packages. Installed research libraries: NumPy 2.2.5, pandas 2.2.3, SciPy 1.15.2, JupyterLab 4.5.6 and nbformat 5.10.4.

**Evidence:** `pip check` reported no broken requirements; 8 Python unit tests and 5 JavaScript tests passed. A non-blocking pip cache permission warning was observed.

**Gap:** CI targets Python 3.11, the local environment uses 3.12, and inherited packages and unpinned transitive dependencies prevent repeatable clean installation.

**Decision D001 (approved by owner, 2026-11-24):** standardize on Python 3.12, isolate the virtual environment, pin all shared dependencies in `requirements.lock`, and preserve licensed data locally. CI now uses Python 3.12. WRDS, other licensed datasets, derived files, browser saves, credentials and logs stay outside Git.

**Implementation evidence:** fresh `.venv` created from the lock using INSTALL.command (the old environment is preserved in `.venv-legacy`). `pip check` passed, 13 Python tests and 5 Web tests passed, and eight explicitly synthetic notebook cells executed with the Python 3.12 kernel. The Jupyter runtime now lives under ignored `.run/`; optional SOCKS support is locked for proxy environments. The web health endpoint returned success and Jupyter returned HTTP 302. Process inspection is restricted, so STOP remains to be verified in the owner's Terminal. No UI or licensed data change. Linux CI remains to be verified remotely.

### G00 — Shared contracts and eligibility (CH27.01–02, SYS.01–02)

Prepare the versioned interfaces before financial modeling: ClientProfile, ConstraintSet, GoalSpec, Card, MarketSnapshot, ForecastBundle, RiskReport, StrategySpec, OptimizerSpec, PortfolioCandidate, Evaluation and Selection. All new contract outputs use a UTC timestamp, run ID, model and schema versions, explicit units, status, warnings and provenance. Keep forecasts separate from risk, strategy from optimizer, and funded weights from futures contracts. Detailed mandate inputs are defined in chapter 01; the legacy demonstration is not a validated financial model.

The eligibility registry covers 22 optimizers, 19 strategies, 13 factors and 10 dynamic methods. It fails closed without independent validation, necessary data, instrument verification, capacity, frequency, and required permissions or margin evidence. The 11 objectives are not methods. No method is approved merely because its interface exists. See `src/riskmatch/contracts.py`, `src/riskmatch/eligibility.py`, `tests/unit/test_contracts.py` and [architecture](../docs/ARCHITECTURE.md).

Collaboration conventions and experiment registry: [CONTRIBUTING](../CONTRIBUTING.md), [experiment registry](../docs/EXPERIMENT_REGISTRY.md). D002: public repository for now; restricted data remain local. Commit c05aaee was confirmed on origin/main and the owner reported green GitHub Actions. Both local ports stopped responding after the owner ran STOP.command. D003: solo development, direct validated pushes permitted; require collaborator access controls, PR review and main protection before collaboration. G00 closed in solo mode. Next: chapter 01 Client Mandate, together with the objective specifications in chapter 12.

In [ ]:
from datetime import datetime, timezone
from riskmatch.contracts import RunMetadata, ForecastBundle, RiskReport
from riskmatch.eligibility import REGISTRY, assess
meta = RunMetadata(as_of=datetime(2026, 11, 24, tzinfo=timezone.utc), run_id='synthetic-g00',
                   model_version='fixture-v1', units=(('return', 'decimal'),), provenance=('synthetic',))
forecast = ForecastBundle(meta=meta, snapshot_id='fixture', instrument_ids=('ETF-A',),
                          expected_returns=(0.01,), horizon_periods=1, frequency='monthly', currency='USD')
risk = RiskReport(meta=meta, snapshot_id='fixture', instrument_ids=('ETF-A',),
                  covariance=((0.002,),), frequency='monthly', currency='USD')
eligibility = assess('OPT01', instruments=frozenset({'equity_etf'}), data=REGISTRY['OPT01'].data,
                     frequency='monthly', capacity_verified=True)
assert not eligibility.eligible and 'method_not_validated' in eligibility.reasons
len(REGISTRY), forecast.meta.status, eligibility.reasons

<a id="chapitre-01"></a>

## 01 — Client Mandate

**Objective:** Define ClientProfile, hard constraints, preferences and feasibility.

Reference: [A–Z Guide, chapter 1](../docs/QUANT_GUIDE_A_Z.md).

Status: chapter 01 implementation, deterministic validation only. Full portfolio feasibility and advisory suitability are not evaluated. See [Client Mandate](../docs/CLIENT_MANDATE.md).

### 01.1 Theory and Assumptions
CH01.01–05: choose CHF/USD; cash, liquid investments, other assets, debt and reserve are distinct. Net worth = cash + investments + other assets − debt; investable wealth = cash + investments − reserve. Capital is allocated from investable wealth. All flows use the base currency; mixed-currency goals require an explicit CHF/USD scenario. Real targets = current purchasing power × (1+inflation)^T. Dates are ACT/365.25. Contributions and withdrawals have separate signs and dates; goals have objective IDs, currency, basis, priority and penalty.

CH01.06: W(T)=C(1+r)^T+Σ CF_i(1+r)^(T−t_i); solve required return by Brent root finding in log(1+r), rejecting ambiguous sign patterns and interim borrowing. Zero-return funding gaps are diagnostics, not a market forecast.

CH01.07–12: explicit loss capacity capped by capital, liquid wealth and minimum zero-return balance over the next 12 months. Stress cap = min(declared stress cap, loss budget, capacity/capital). Risk willingness and required risk remain distinct. Experience and permissions are independent. Hard limits never relax silently; missing optional limits are null. Structural conflicts are returned explicitly.

CH01.13–15: version 2.0.0 payload, synthetic fixture, API preview, before/after review and explicit confirmation. Restricted real data remain outside Git; older minimal saves retain legacy scope. The USD demo cannot evaluate CHF or unmodeled hard constraints. No model approval is implied.

In [ ]:
import json
from riskmatch.profiling.mandate import validate_mandate
profile = json.loads((root / 'configs' / 'mandate-example.json').read_text())
mandate_report = validate_mandate(profile)
assert mandate_report['status'] == 'structurally_valid'
assert mandate_report['net_worth'] == 180000 and mandate_report['investable_wealth'] == 130000
assert mandate_report['goals'][0]['zero_return_gap'] == 20000
{k: mandate_report[k] for k in ('net_worth', 'investable_wealth', 'monthly_surplus', 'effective_stress_limit', 'goals', 'combined_required_return')}

### 01.2 Validation
- [x] Analytical case and synthetic fixture (tests/unit/test_mandate.py)
- [x] Dated flows and explicit scenario inputs; no market observations
- [x] Edge cases, CHF/USD conversion and constraint sensitivity
- [ ] Out-of-sample market results are not part of mandate validation

### 01.3 Promotion to the Application
- [x] Extract functions into src/riskmatch/profiling/mandate.py
- [x] Add unit, loopback API and Web App confirmation tests
- [x] Connect API/UI and show limitations
- [ ] Owner visual confirmation and GitHub push (solo mode)

<a id="chapitre-02"></a>

## 02 — Behavioral Discovery

**Objective:** Version cards, responses, coverage, score and sensitivity.

Reference: [A–Z Guide, chapter 20](../docs/QUANT_GUIDE_A_Z.md).

Status: implemented experimental Discovery v2 (CH02.01–15 and CH02.21). Human validation CH02.16–20 remains open. See [behavioral protocol](../docs/RISKMATCH_BEHAVIORAL.md) and [draft pilot](../docs/BEHAVIORAL_PILOT.md).

### 02.1 Theory and Assumptions
Risk tolerance, loss sensitivity, path tolerance, ambiguity, time preference, style, confidence and consistency are separate constructs. Framing, anchoring and recency are research hypotheses, not individual diagnoses. The 60-card bank contains 16 lottery, 12 paired frames, 12 trajectories, 8 ambiguity/time, 8 style and 4 repeat items. Sessions select 8/6/6/4/4/2, balance placement, preserve matched pairs, store seed/version and record response locale.

Only 20 lottery/frame/trajectory items enter the exploratory score. Let k be prudent resolved answers and n resolved scored answers. Score = floor(1+29 k/n+0.5) if n/20≥0.70; otherwise unavailable. Missing sensitivity = [round(1+29k/20), round(1+29(k+20−n)/20)]. Compare canonical choices after swaps; report repeat differences and framing differences separately. The range is not a confidence interval. Confidence is optional self-report. No score changes the financial loss budget.

Utility exploration: CRRA (w^(1−γ)−1)/(1−γ), log(w) at γ=1; prospect value x^α or −λ_loss|x|^α. Objective probabilities, fixed logistic temperature and small grid; not a calibrated person-level risk parameter. Separate from score and optimizer λ.

Pilot readiness: synthetic complete anchor panel, difficulty, corrected discrimination, test-retest bootstrap, exploratory omega and eigenvalues, person-level 60/20/20 splits, logistic anchor proxy, language/order DIF and independent choice log-loss. Not human evidence or latent IRT calibration. Actual pilot, interviews, expert review and validation remain CH02.16–20.

In [ ]:
from riskmatch.behavioral.discovery import bank, session, score as discovery_score
from riskmatch.behavioral.utility import explore_likelihoods
from riskmatch.behavioral.pilot_analysis import synthetic_panel, psychometrics, logistic_readiness
discovery = session(42, 'CHF')
prudent = {c['id']: c['prudent_response'] for c in discovery['cards'] if c['scored']}
risky = {key: ('dislike' if value == 'like' else 'like') for key, value in prudent.items()}
assert len(bank()) == 60 and len(discovery['cards']) == 30
assert discovery_score(42, 'CHF', prudent)['score'] == 30
assert discovery_score(42, 'CHF', risky)['score'] == 1
assert discovery_score(42, 'CHF', dict(list(prudent.items())[:13]))['score'] is None
utility_demo = explore_likelihoods(42, 'CHF', prudent)
pilot = synthetic_panel(n=120, items=20, seed=42)
synthetic_psychometrics = psychometrics(pilot, bootstrap=50)
synthetic_logistic = logistic_readiness(pilot)
{'score': discovery_score(42, 'CHF', prudent)['score'], 'utility_status': utility_demo['status'], 'synthetic_retest': synthetic_psychometrics['test_retest'], 'synthetic_test_loss': synthetic_logistic['independent_choice_test_loss'], 'human_validation': False}

### 02.2 Validation
- [x] Bank quotas, deterministic placement, pair equivalence and extreme/missing/paired response fixtures
- [x] Synthetic person-level train/validation/test split; no response-level leakage
- [x] Score sensitivity, utility likelihood and independent synthetic baseline comparison
- [ ] Actual pilot validity, test-retest, interviews and independent review remain open

### 02.3 Promotion to the Application
- [x] Extract functions into src/riskmatch/behavioral
- [x] Unit, HTTP and web flow tests
- [x] Connect API/UI: consent, randomized session, score, separate dimensions, undo and withdrawal
- [x] Show the Discovery page before mandate confirmation, with a direct link to complete the mandate before questionnaire access
- [ ] Owner visual confirmation and GitHub push (solo mode)

<a id="chapitre-03"></a>

## 03 — US Universe

**Objective:** Build the ETF security master: ticker, currency, class and dates.

**Fixed-income universe:** only T-Bills, Treasury Notes, Treasury Bonds, TIPS and bond ETFs with verified Treasury-only point-in-time holdings; exclude corporate, agency, municipal, MBS, foreign sovereign and mixed ETFs. UNIV.01 is a hard allocation filter.

Reference: [A–Z Guide, chapter 2](../docs/QUANT_GUIDE_A_Z.md).

Status: candidate security master and local coverage interface implemented. See [US Universe](../docs/US_UNIVERSE.md). The 22 ETF candidates are not verified investable portfolios; historical liquidity and Treasury-only eligibility remain gated. CH03.05–08 expansion contracts prepared, not activated.

### 03.1 Theory and Assumptions
The issuer-sourced catalog consists of 17 equity/real estate ETFs, 5 Treasury-only candidate ETFs, and a separate USD cash ledger. Permanent IDs are independent of ticker. Current issuer benchmark/inception metadata is not a historical investability record. Use date-valid ticker aliases and observed/available/valid-until listing, expense, ADV and AUM records; reject ambiguity, future or stale records. Provisional liquidity screen: USD 1m ADV and USD 100m AUM; no missing values treated as zero. Verify full historical Treasury holdings and exclude non-Treasury debt.

Coverage uses distinct observed source-grid sessions after the inception/listing boundary, not an exchange-calendar guarantee. Coverage ratio = observed/expected source-grid dates. No forward fill, prelaunch history or lookahead. A past local receipt summary is reported separately and is not newly observed prices. DTB3 is a discount yield proxy, not a cash return. Benchmarks are defined separately from executable fund proxies.

Extension designs: PIT CRSP PERMNO ranking (300–3000), direct Treasury CUSIP/type/maturity, per-expiry futures specs and USD/CHF/equity/rate hedge map. No actual extension eligibility before G06 and verified sources. No options enabled. Updated local audit: 12 adjusted daily ETF series through Nov 2026, all 22 ETF daily/monthly CRSP histories through 2025, 1,000 preliminary stock candidates ranked at 2025-12-31. All raw/extracted licensed values remain local. Public Treasury auction identities and five recent issuer holdings snapshots were acquired; the strict current Treasury-only rule blocks all five because they report money-market fund shares. No automatic look-through or historical backfill. See acquisition and monitoring plans.

In [ ]:
from datetime import date
from riskmatch.market_data.security_master import load_catalog, read_adjusted_csv, coverage_matrix, assess_etf
catalog, instruments = load_catalog()
assert len(instruments) == 27
from riskmatch.market_data.local_sources import membership_summary
if __import__("os").environ.get("RISKMATCH_SHOW_LOCAL_COVERAGE") == "1":
    print(membership_summary(date.today(), root))
synthetic_prices = read_adjusted_csv(root / 'configs' / 'universe-coverage-example.csv')
coverage = coverage_matrix(instruments, synthetic_prices, source='synthetic', as_of=date(2018,6,20))
assert coverage['XLC']['prelaunch_rejected'] == 2 and coverage['XLC']['missing_fraction'] == 0.5
assert all(not assess_etf(i, date(2026,11,24))['eligible'] for i in instruments)
{'catalog_version': catalog['version'], 'candidates': len(instruments), 'synthetic_coverage': {k: coverage[k] for k in ('SPY','IEF','XLC')}}

### 03.2 Validation
- [x] Synthetic coverage fixture, including deliberately invalid prelisting quotes
- [x] Ticker reuse, publication cutoff, stale records and Treasury-only fail-closed tests
- [x] Benchmark/proxy distinction, synthetic stock and per-expiry contract schemas
- [ ] Actual historical raw coverage, liquidity, holdings and extension verification remain open

### 03.3 Promotion to the Application
- [x] Extract functions into src/riskmatch/market_data/security_master.py
- [x] Unit, HTTP and Web filter tests
- [x] Read-only Universe page with candidate/source/coverage/exclusion states
- [ ] Owner visual confirmation (solo mode)

Local research evidence now covers 336 monthly stock rankings, 132,278 volume-matched ETF observations and two optional USD spot series (Bitcoin and Ethereum). The strict Treasury rule excludes all 138 date-specific N-PORT filings examined. Holdings between disclosures and the 2026 CRSP refresh remain open. The weekly CFTC records for ten futures families and a one-day ES/NQ contract sample cannot support a roll backtest.

The 2026 refresh covers 22 ETFs on 225 sessions and 955 of the 1,000 stock candidates on all those sessions. The futures archive contains 1,470 family/month pairs through March 2024. Compare coverage at the decision date. A futures price with a month label does not establish settlement or contract identity before exchange launch. See the verified coverage table in the acquisition plan.

FedInvest has 288,594 positive issue-day valuation observations across 694 dates, including all 218 reference month-ends and all 497 recent curve-calendar dates through November 20, 2026. The latest separately verified valuation is November 23. Coverage on a calendar date requires at least one matched issue. Individual total returns require separate validation.

The active universe contains zero futures families. Retain the ten historical families for research only. None has verified per-expiry coverage through November 2026. Futures-based USD/CHF, equity and rates hedges are unavailable. Futures history is outside the closure criteria for Chapters 03 and 04.

The frozen catalog has 27 listed products. The research-window build reconciles the year-end closes, retains missing next-month stock returns, and separates bill mark intervals from nominal coupon schedules. Read the acquisition plan before selecting a sample or combining return types.


<a id="chapitre-04"></a>

## 04 — Sources and Licenses

**Objective:** Catalog providers, licenses and data availability dates.

Reference: [A–Z Guide, chapter 3](../docs/QUANT_GUIDE_A_Z.md).

Status: source inventory, local CRSP audit, immutable public downloads, source readiness interface and optional update adapters implemented. Actual entitlement confirmation, full PIT histories, secondary Treasury prices, historical ETF holdings and per-expiry futures remain open. See [acquisition plan](../docs/DATA_ACQUISITION_PLAN.md) and [monitoring plan](../docs/DATA_MONITORING_PLAN.md).

### 04.1 Theory and Assumptions
Maintain a 16-category source registry, explicit provider/version/as_of contracts and availability diagnostics. CRSP CIZ coverage is local research evidence, not historical investability. Distinguish observation date, publication time, receipt timestamp, source revision and rights. Never merge daily adjusted values and CRSP returns without reconciliation. Store immutable raw snapshots and SHA-256 receipt hashes; large restricted source audits use explicit local streaming, not CI or notebook outputs.

Public sources acquired: government Treasury auction records, 15 FRED current-vintage series, five issuer holdings snapshots and a local Treasury identity master. FRED current vintage is not ALFRED historical availability. Treasury auction prices are not secondary daily prices. FedInvest dated security-level government valuation marks are now partially acquired at zero cost; backfill and execution-price validation remain pending. Quarterly SEC holdings are stored by accession. The local audit contains 6,110 positions for five funds and tracks incomplete timestamp coverage. Current fund holdings contain ancillary money-market shares; do not silently relax direct Treasury-only policy.

Free/paid monitoring: WRDS for research, optional EODHD authorized token for ETF update batches, specialist futures/Treasury feeds or institutional cross-asset provider. No key, paid subscription or schedule configured. Missing sources fail closed. The following CI cell uses only public configuration and a mocked unavailable provider.

In [ ]:
from riskmatch.market_data.source_catalog import catalog, ProviderRequest, request_snapshot
source_catalog = catalog()
assert len(source_catalog['entries']) == 16
offline = request_snapshot(ProviderRequest('not_subscribed', 'fixture-v1', date(2026,11,24)), {})
assert offline['status'] == 'provider_unavailable'
{'catalog_version': source_catalog['version'], 'source_categories': len(source_catalog['entries']), 'offline_provider': offline['status']}

from riskmatch.market_data.local_sources import audit_available_for_as_of
import json
import os
coverage_file = root / 'data/processed/universe/research_extensions.json'
if os.environ.get("RISKMATCH_SHOW_LOCAL_COVERAGE") == "1" and coverage_file.is_file():
    research_coverage = audit_available_for_as_of(json.loads(coverage_file.read_text()), date.today())
else:
    research_coverage = None
print('Local research coverage:', ', '.join(sorted(k for k in (research_coverage or {}) if k != 'audited_on')) or 'not yet acquired')

from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as empty_project:
    unavailable = request_snapshot(ProviderRequest("fixture", "v1", date(2026, 1, 1)), root=Path(empty_project))
    assert unavailable["status"] == "inventory_unavailable"


### 04.2 Validation
- [x] Source registry and explicit unavailable-provider fixture prepared
- [x] Local CRSP daily/monthly and public source acquisition audits performed
- [ ] Full entitlement, publication/revision, provenance and historical holdings validation pending
- [ ] Complete allocation-quality data gate pending

### 04.3 Promotion to the Application
- [x] Source readiness module, local audits and public downloader scripts
- [x] Synthetic source receipt / unavailable provider tests prepared
- [x] Sources and updated Universe API/UI; no allocation approved
- [ ] Owner visual confirmation and license gate

The 11 WRDS deliveries have been scanned. The official monthly debt statements, TIPS ratios and two distinct price archives have separate local provenance. See the [verified delivery and remaining gaps](../docs/DATA_ACQUISITION_PLAN.md).

Run `scripts/audit_research_extensions.py` after downloads. The Sources page reads this dated local summary. For fundamentals, join the original EDGAR accession and acceptance time; preserve the XBRL tag, unit and duration. Keep the current stock refresh and all licensed issuer mappings local.

Local metadata receipts support exact-version lookups with a UTC verification cutoff. See the [source quality report and field dictionary](../docs/SOURCE_REVIEW.md). Institutional rights and final method approval remain open.

<a id="chapitre-05"></a>

## 05 — Point-in-Time Ingestion

**Objective:** Normalize snapshots, corporate actions, calendars and leakage tests.

Reference: [A–Z Guide, chapter 3.1](../docs/QUANT_GUIDE_A_Z.md).

Status: technical pipeline implemented. Local Parquet partitions remain research-only until source and historical availability validation.

### 05.1 Theory and Assumptions
Observation date and availability timestamp are distinct. The decision cutoff is UTC; unknown release times and late revisions are excluded. Features use explicit lag and lookback, with no forward fill. The expanded local inputs include factors, original macro vintages, Compustat, CCM, filing acceptance references, recent stock ticker bars, Treasury marks, TIPS indexation, dated fees and liquidity. See [pipeline design](../docs/POINT_IN_TIME.md).

In [ ]:
from datetime import date, datetime, timezone
from riskmatch.market_data.point_in_time import Observation, as_of_rows, equity_close, lagged_feature
d1, d2 = date(2025, 1, 2), date(2025, 1, 3)
cutoff = equity_close(d2, 16)
original = Observation('SYNTH', 'close', d1, equity_close(d1, 16), 'v1', 100.0, 'USD/share')
correction = Observation('SYNTH', 'close', d1, datetime(2025, 1, 6, tzinfo=timezone.utc), 'v2', 104.0, 'USD/share')
unknown = Observation('SYNTH', 'close', d2, None, 'v1', 110.0, 'USD/share')
assert as_of_rows([original, correction, unknown], cutoff) == (original,)
feature = lagged_feature([original, correction, unknown], cutoff, instrument='SYNTH', field='close', session_days=[d1,d2], lag=1, lookback=1, unit='USD/share')
assert feature['value'] == 100.0

### 05.2 Validation
- [x] Analytical case or fixture
- [x] No look-ahead leakage on synthetic revision test
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 05.3 Promotion to the Application
- [x] Extract functions into src/riskmatch
- [x] Add tests
- [x] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-06"></a>

## 06 — Returns and FX

**Objective:** Calculate simple/log/total returns, FX and reconcile adjusted prices.

Reference: [A–Z Guide, chapter 3.1](../docs/QUANT_GUIDE_A_Z.md).

Status: technical return engine and local research partitions implemented; historical use awaits source approval.

### 06.1 Theory and Assumptions
Simple return, log return, split/dividend total return, compounding, nominal/real return, FX, cash accrual, bond dirty-price return, futures cash P&L and TWR/XIRR are distinct conventions. See [Returns methodology](../docs/RETURNS.md).

In [ ]:
from math import isclose, expm1
from datetime import date, timedelta
from riskmatch.market_data.returns import total_return, compound, log_return, fx_return, bond_return, cash_accrual, time_weighted_return, money_weighted_return, tips_return
r = total_return(100, 55, split=2, distribution=2)
assert isclose(r, .12)
assert isclose(compound([.1, -.05]), expm1(log_return(.1)+log_return(-.05)))
assert isclose(fx_return(.1, .9, .95), fx_return(.1, 1/.9, 1/.95, inverse=True))
assert isclose(bond_return(98, 99, 1, .2, 2), 101.2/99-1)
assert isclose(tips_return(100, 100, 0, 0, 1, 1.02), .02)
assert isclose(time_weighted_return([100,165,181.5],[55,0]), .21)
d = date(2025,1,1)
assert isclose(money_weighted_return([(d,-100),(d+timedelta(days=365),110)]), .10)
assert isclose(cash_accrual(.05,d,d+timedelta(days=3)),.05*3/360)

### 06.2 Validation
- [x] Analytical case or fixture
- [x] Missing intervals and future availability controls
- [x] CRSP daily/monthly and SOFR reference reconciliation
- [ ] Out-of-sample results where applicable

### 06.3 Promotion to the Application
- [x] Extract functions into src/riskmatch
- [x] Add tests
- [x] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-07"></a>

## 07 — Descriptive Statistics

**Objective:** Means, volatility, covariance, correlation and charts.

Reference: [A–Z Guide, chapter 4](../docs/QUANT_GUIDE_A_Z.md).

Status: descriptive research engine implemented; actual historical data approval remains pending.

### 07.1 Theory and Assumptions
Sample variance uses n−1. Annualization depends on frequency. Covariance matrices align exact intervals and discard jointly incomplete rows. Trailing windows and a fixed training split prevent future leakage. The Universe tab presents a client catalogue (name, ticker and source), dedicated statistical profiles with explained charts, and same-frequency comparisons across sources. Acquisition evidence remains in Sources. See [descriptive methodology](../docs/DESCRIPTIVE_STATISTICS.md).

In [ ]:
import numpy as np
from riskmatch.analytics.descriptive import summarize, covariance_matrix, portfolio_variance, distribution, rolling_statistics, training_transform, time_diagnostics
r = np.array([[.01,.02],[.02,.04],[.03,.06]])
summary = summarize(r[:,0],252)
assert np.isclose(summary['variance'], .0001)
matrix = covariance_matrix(r)
assert matrix['psd'] and np.isclose(portfolio_variance(matrix['covariance'], [.5,.5]), .000225)
assert np.isclose(rolling_statistics(r[:,0],r[:,1],3,252)[-1]['beta'], .5)
assert sum(x[2] for x in distribution(r[:,0])['histogram']) == 3
assert np.isclose(training_transform([-.1,.1],[2])[0],training_transform([-.1,.1],[2,999])[0])

### 07.2 Validation
- [x] Analytical case or fixture
- [x] Trailing and fixed-training leakage tests
- [x] Covariance and portfolio variance checks
- [ ] Out-of-sample results where applicable

### 07.3 Promotion to the Application
- [x] Extract functions into src/riskmatch
- [x] Add tests
- [x] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-08"></a>

## 08 — Return Forecasting

**Objective:** Historical, EWMA, CAPM, Black-Litterman and uncertainty.

Reference: [A–Z Guide, chapter 4](../docs/QUANT_GUIDE_A_Z.md).

Status: corrected corporate-action returns and bounded simple-return forecasts; 1,008 of 1,474 items computed all ten core models, 464 have separate conditional yield/short-history estimates, two remain unavailable. A 575-asset monthly USD total-return cohort at 2025-12-31 is separate historical research. Model approval and dependent factor/cash/portfolio work remain pending. Reproduce local batch with `.venv/bin/python scripts/build_forecast_universe.py --as-of 2026-11-24 --workers 4`; results remain local. Different horizons, return targets and cutoffs are not directly comparable.

### 08.1 Theory and Assumptions
Use nonoverlapping one-period decimal returns, strictly lagged features, chronological 60/20/20 splits, training-only scaling and validation-only tuning. Empirical intervals use validation residual quantiles. CAPM/multifactor and the no-view Black-Litterman prior are synthetic references pending CH10/15 and G02/G07. See [forecast methodology](../docs/RETURN_FORECASTING.md).

In [ ]:
from riskmatch.analytics.forecasting import forecast, factor_reference
from datetime import date, timedelta
import numpy as np
rng = np.random.default_rng(42)
r = rng.normal(0.001, 0.02, 220)
dates = [date(2020, 1, 1) + timedelta(days=i) for i in range(len(r))]
f08 = forecast(r, dates, instrument='synthetic', frequency='daily', currency='USD', kind='total_return', data_version='synthetic-v1', as_of=dates[-1], include_ml=True)
assert {'random_forest', 'xgboost', 'neural_network'} <= f08['models'].keys()
assert not f08['approved'] and f08['bundle']['horizon_periods'] == 1
factors = rng.normal(0, 0.02, (220, 3))
rf = np.full(220, .001)
refs = factor_reference(rf + .002 + factors @ np.array([1.2, .4, -.3]), factors, rf, [.5, .5], np.eye(2)*.02)
assert np.allclose(refs['multifactor']['beta'], [1.2, .4, -.3])
f08['selected_model'], {k: v['test']['mae'] for k, v in f08['models'].items()}

### 08.2 Validation
- [x] Analytical case or fixture
- [x] No look-ahead leakage in the one-period experimental split
- [ ] Full benchmarks and sensitivity tests (cash, factors and windows pending)
- [x] Held-out synthetic results for the implemented models

### 08.3 Promotion to the Application
- [x] Extract functions into src/riskmatch
- [x] Add tests
- [x] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-09"></a>

## 09 — Volatility and Covariance

**Objective:** EWMA, GARCH, shrinkage, DCC; PSD and OOS comparison.

Reference: [A–Z Guide, chapter 4](../docs/QUANT_GUIDE_A_Z.md).

Status: experimental volatility and covariance engine implemented, with catalogue-wide research and common-interval covariance cohorts. Formal approval remains blocked by G04 and the external factor chapter.

### 09.1 Theory and Assumptions
Use exact aligned intervals, common return kind and currency. Estimate historical, EWMA and conditional GARCH/EGARCH/GJR-GARCH variance on chronological splits; compare sample, EWMA, Ledoit-Wolf, OAS and training PCA factor covariance. DCC uses standardized training innovations and positive-semidefinite correlation. See [risk methodology](../docs/VOLATILITY_COVARIANCE.md).

In [ ]:
from riskmatch.analytics.volatility import volatility_models, covariance_models, dcc_correlation
import numpy as np
from datetime import date, timedelta
rng = np.random.default_rng(42)
z09 = rng.multivariate_normal([0,0], [[1,.5],[.5,1]], size=700)
h09 = np.full(2,.0004); r09 = []
for shock in z09:
    e09 = np.sqrt(h09)*shock; r09.append(e09)
    h09 = .00002 + .12*e09**2 + .83*h09
r09 = np.array(r09); d09 = [date(2020,1,1)+timedelta(days=i) for i in range(len(r09))]
vol09 = volatility_models(r09[:,0],d09,252)
cov09 = covariance_models(r09,252)
dcc09 = dcc_correlation(r09)
assert all(v['diagnostics']['before']['psd'] for v in cov09['models'].values())
assert dcc09['covariance_diagnostics']['psd']
vol09['selected_model'], cov09['selected_model']

### 09.2 Validation
- [x] Analytical case or fixture
- [x] No look-ahead leakage in train/validation selection
- [x] Benchmarks and sensitivity tests
- [x] Experimental out-of-sample results; financial approval pending

### 09.3 Promotion to the Application
- [x] Extract functions into src/riskmatch
- [x] Add tests
- [x] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-10"></a>

## 10 — Factors and Regressions

**Objective:** 13 factors, exposures, scores, OLS, PCA and attribution.

Reference: [A–Z Guide, chapter 8](../docs/QUANT_GUIDE_A_Z.md).

Status: experimental factor research implemented on the full catalogue; realized published factor exposures, dated internal scores, macro proxies and analytical Treasury references. Data approval and executable factor-portfolio validation remain pending.

### 10.1 Theory and Assumptions
Map 13 factors to published returns, dated scores or explicitly labelled proxies. Use strict filing-date availability, next-period outcomes, train-only OLS/HAC and ridge tuning, common-interval covariance and attribution. See [factor methodology](../docs/FACTORS_REGRESSIONS.md).

In [ ]:
from riskmatch.analytics.factors import regression, zscore, pca_diagnostics, TAXONOMY
from datetime import date, timedelta
import numpy as np
rng = np.random.default_rng(42)
x10 = rng.normal(0,.02,(180,3))
y10 = .001 + x10 @ np.array([1.2,.4,-.3]) + rng.normal(0,.001,180)
d10 = [date(2020,1,1)+timedelta(days=i) for i in range(180)]
fit10 = regression(y10,x10,d10,['Mkt-RF','SMB','HML'],'daily')
a10 = fit10['attribution']
assert abs(a10['alpha'] + sum(a10['mean_factor_contribution'].values()) + a10['mean_residual'] - a10['mean_realized_excess']) < 1e-10
assert abs(fit10['beta']['Mkt-RF']-1.2)<.03 and len(TAXONOMY)==13
fit10['beta'], pca_diagnostics(x10)['explained_variance_ratio']

### 10.2 Validation
- [x] Analytical case or fixture
- [x] Train-only exposure and hyperparameter tests
- [x] Benchmarks and sensitivity tests
- [x] Experimental held-out attribution; actual PIT approval remains pending

### 10.3 Promotion to the Application
- [x] Extract functions into src/riskmatch
- [x] Add tests
- [x] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-11"></a>

## 11 — Tail Risk

**Objective:** VaR, ES, EVT, copulas, drawdown and validation.

Reference: [A–Z Guide, chapter 10](../docs/QUANT_GUIDE_A_Z.md).

Status: executed prototype, pending validation. Demo data are synthetic.

### 11.1 Theory and Assumptions
Loss L = −r, in decimal per observed period. Positive values are losses. VaR is the empirical inverse CDF at confidence alpha, ES integrates the worst 1−alpha mass with fractional boundary weights. Gaussian and simulated normal, fitted Student-t, and GPD peaks over threshold are research estimates. Minimum five expected tail observations; POT needs 30 exceedances and finite ES. The 70/30 chronological holdout reports Kupiec and independence diagnostics; sparse exceptions have low power. Drawdown compounds wealth relative to prior peak. Copulas use training ranks and a fixed t df=5 sensitivity, exact common periods, equal-weight scenarios and additive ES contributions. VaR is never a guaranteed maximum.

In [ ]:
import numpy as np
from riskmatch.analytics.tail_risk import profile, dependence, empirical
rng = np.random.default_rng(11)
fixture = rng.standard_t(5, size=(600, 3)) * .015
tail_fixture = profile(fixture[:,0], range(600))
dependence_fixture = dependence(fixture)
assert np.isclose(sum(dependence_fixture['es_contributions']['contributions']), dependence_fixture['es_contributions']['es'])
assert np.isclose(empirical([0,1,2,2,8], .7)['es'], 6)
{k: list(v) for k,v in tail_fixture['models'].items()}

### 11.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 11.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-12"></a>

## 12 — Financial Objectives

**Objective:** Formalize the 11 objectives, units, constraints and loss function.

**Order:** specify GoalSpec and OBJxx.01 with the mandate (G01); evaluate goals quantitatively after their required data and simulations.

Reference: [A–Z Guide, chapter 5](../docs/QUANT_GUIDE_A_Z.md).

Status: implemented experimental goal evaluation with synthetic fixture and local historical reference comparisons. Independent approval and instrument-specific coverage remain pending.

### 12.1 Theory and Assumptions
Eleven typed objectives map to functions and independent eligibility constraints. Wealth uses dated contributions, withdrawals, liabilities and priority-ordered funding goals. Evaluate P(W at due date >= target), expected deficit, normalized shortfall and priority-weighted penalty; real targets use the mandate inflation scenario. Historical reference portfolios use a six-month moving-block bootstrap on aligned monthly returns, monthly rebalancing and chronological 80/20 split. No borrowing, no transaction costs, no certified hedge or liability duration match; total-minus-price return is only a distribution proxy.

In [ ]:
import json, numpy as np
from pathlib import Path
from riskmatch.analytics.objectives import evaluate, SPECS
fixture_path = next(p for p in [Path('configs/mandate-example.json'), Path('../configs/mandate-example.json')] if p.exists())
m = json.loads(fixture_path.read_text())
m.update(as_of='2026-01-01', horizon_date='2027-01-01', contribution_monthly=0)
m['goals'][0].update(amount=120000, due_date='2027-01-01')
z = np.zeros((200,12))
goal_fixture = evaluate(m,z,z)
assert goal_fixture['goals'][0]['expected_deficit'] == 20000
assert len(SPECS) == 11
goal_fixture['goals'][0]

### 12.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 12.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-13"></a>

## 13 — Baselines

**Objective:** Cash, 1/N, 60/40 and market benchmark.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: executed prototype, pending validation. Demo data are synthetic.

### 13.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
from riskmatch.prototype import ClientProfile, candidates
[(p['name'], p['metrics']['cagr']) for p in candidates(ClientProfile(100000, 10, 1))]

### 13.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 13.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-14"></a>

## 14 — Mean-Variance

**Objective:** Minimum variance, maximum Sharpe, maximum utility, frontier and constraints.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 14.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 14.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 14.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-15"></a>

## 15 — Black-Litterman

**Objective:** Prior, views, confidence, posterior and allocation.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 15.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 15.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 15.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-16"></a>

## 16 — Risk Parity and Hierarchical Methods

**Objective:** RP, HRP, HERC, diversification and risk contributions.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 16.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 16.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 16.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-17"></a>

## 17 — Downside Optimization

**Objective:** CVaR, Sortino, Omega, drawdown, tracking error and IR.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 17.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 17.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 17.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-18"></a>

## 18 — Advanced Optimization

**Objective:** Robust, resampled, DRO, Kelly, entropy and diagnostics.

Reference: [A–Z Guide, chapter 6](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 18.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 18.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 18.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-19"></a>

## 19 — Passive and Active Strategies

**Objective:** Passive, long-only, smart beta, core-satellite and factors.

Reference: [A–Z Guide, chapter 7](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 19.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 19.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 19.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-20"></a>

## 20 — Alternative Strategies

**Objective:** Long/short, neutral, macro, CTA, arbitrage, carry, momentum and others.

Reference: [A–Z Guide, chapter 7](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 20.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 20.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 20.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-21"></a>

## 21 — US Treasuries

**Objective:** T-Bills, Notes, Bonds, TIPS: cash flows, clean/dirty prices, duration, convexity, DV01 and curve risk.

Build nominal/real curves, slope/curvature (FAC11 Treasury Curve), breakeven and key-rate DV01. No corporate credit models.

Reference: [A–Z Guide, chapter 4](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 21.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 21.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 21.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-22"></a>

## 22 — Futures

**Objective:** Expiries, multipliers, margin, collateral and roll.

Reference: [A–Z Guide, chapter 3.1](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 22.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 22.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 22.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-23"></a>

## 23 — Walk-Forward Backtesting

**Objective:** Chronology, benchmarks, fees, delayed execution and ledger.

**Order:** BT00.01–06 (G04) before advanced strategies; ETF/cash ledger, benchmarks, reconciliation and OOS folds. Extend Treasuries/futures (G07) after chapters 21–22.

Reference: [A–Z Guide, chapter 9](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 23.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 23.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 23.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-24"></a>

## 24 — Stress and Monte Carlo

**Objective:** Scenarios, terminal wealth distribution, goals, flows and uncertainty.

Reference: [A–Z Guide, chapter 11](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 24.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 24.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 24.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-25"></a>

## 25 — Improvement, Matching and Comparison

**Objective:** Separate scientific portfolio improvement from personalized selection.

### 25.A — Scientific Improvement (CH25.15–21, IMP.01–02)
Compare variants against identical baselines; measure ablations, costs, uncertainty and robustness. Publish validated candidates only.

### 25.B — RiskMatch Explore (CH25.01–14, MATCH.01)
Filter mandate constraints, construct a Pareto ranking, allow likes/dislikes and comparisons, and confirm a paper selection. Test MVP Explore on baselines (MVP.01–04) at G05; final matching waits for IMP.02.

Reference: [A–Z Guide, chapter 12](../docs/QUANT_GUIDE_A_Z.md).

Status: executed prototype, pending validation. Demo data are synthetic.

### 25.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
from riskmatch.prototype import ClientProfile, candidates
[(p['name'], p['stress_loss']) for p in candidates(ClientProfile(100000, 10, .1))]

### 25.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 25.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-26"></a>

## 26 — Dynamic Management

**Objective:** 10 dynamic models, rebalancing, monitoring and costs.

Reference: [A–Z Guide, chapter 13](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 26.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 26.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 26.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation

<a id="chapitre-27"></a>

## 27 — Integration and Validation

**Objective:** Reports, exports, UI tests, security, independent validation and paper trading.

Reference: [A–Z Guide, chapter 17](../docs/QUANT_GUIDE_A_Z.md).

Status: planned. Demo data are synthetic.

### 27.1 Theory and Assumptions
Define notation, frequency, units, constraints and limitations.

In [ ]:
# TODO: implement the experiment and document assumptions

### 27.2 Validation
- [ ] Analytical case or fixture
- [ ] No look-ahead leakage
- [ ] Benchmarks and sensitivity tests
- [ ] Out-of-sample results where applicable

### 27.3 Promotion to the Application
- [ ] Extract functions into src/riskmatch
- [ ] Add tests
- [ ] Connect API/UI and show limitations
- [ ] PR review and documentation